# 8교시. 자동화의 마지막 단계는 사람의 확인

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leecks1119/document_ai_lecture/blob/document_ai_lecture_2026/colab/08_business_application.ipynb)

**목표:** 전체 흐름을 점검하고 사람 검토가 있는 적용 카드를 씁니다.

**결과물:** `business_application_card.md`

- 기본 경로는 API 키와 OCR 모델 다운로드가 필요 없습니다.
- 선택 실습은 기본값이 `False`입니다.
- 수업이 지정한 공개 실물·합성 샘플만 사용합니다.


In [ ]:
import platform
import sys
from pathlib import Path

OUTPUT_DIR = Path("course_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Output:", OUTPUT_DIR.resolve())


In [ ]:
from copy import deepcopy

SAMPLE_RECEIPT = {'document_type': 'receipt', 'store_name': '샘플문구점', 'date': '2026-07-27', 'total_amount': 5000, 'items': [{'name': '연필', 'quantity': 2, 'unit_price': 1000, 'line_total': 2000}, {'name': '노트', 'quantity': 1, 'unit_price': 3000, 'line_total': 3000}], 'source_mode': 'mock'}

def validate_receipt(data):
    errors = []
    for field in ("store_name", "date", "total_amount", "items"):
        if data.get(field) in (None, "", []):
            errors.append(f"필수값 누락: {field}")
    item_sum = sum(
        item["line_total"] for item in data.get("items", [])
    )
    if data.get("total_amount") != item_sum:
        errors.append("품목 합계와 총액이 다릅니다.")
    return {"valid": not errors, "warnings": [], "errors": errors}

def run_smoke_test():
    missing = deepcopy(SAMPLE_RECEIPT)
    missing["store_name"] = None
    wrong_total = deepcopy(SAMPLE_RECEIPT)
    wrong_total["total_amount"] = 6000
    return {
        "mock_path_works": SAMPLE_RECEIPT["source_mode"] == "mock",
        "normal_result_is_valid": validate_receipt(SAMPLE_RECEIPT)["valid"],
        "missing_required_is_blocked": not validate_receipt(missing)["valid"],
        "wrong_total_is_blocked": not validate_receipt(wrong_total)["valid"],
    }


## 핵심 3개

1. 자동화 후보는 반복량·오류 영향·예외 빈도로 봅니다.
2. 개인정보·외부 전송·보존·삭제를 확인합니다.
3. 사람의 최종 승인과 수정 절차를 정합니다.


In [ ]:
smoke_result = run_smoke_test()
assert all(smoke_result.values())
print(smoke_result)


## 실습. 한 장짜리 적용 카드


In [ ]:
card = '''# 문서 자동화 업무 적용 카드

| 항목 | 작성 내용 |
| --- | --- |
| 입력 문서 | 사내 비용 처리용 영수증 |
| 필요한 추출 필드 | 상호명, 날짜, 품목, 총액 |
| 틀렸을 때의 영향 | 총액 오류 시 정산 금액이 달라짐 |
| 사람 검토자 | 비용 처리 담당자 |
| 저장 형식과 위치 | 승인 후 CSV, 승인된 저장소 |
| 원본·결과 삭제 시점 | 조직의 보존 기준에 따름 |

## 적용 전 확인

- [x] 합성 문서로 기능을 점검했다.
- [ ] 실제 개인정보의 외부 전송 승인을 확인한다.
- [ ] 최종 승인자와 반려 절차를 확인한다.
'''

output_path = OUTPUT_DIR / "business_application_card.md"
output_path.write_text(card, encoding="utf-8")
print(card)
print("저장 완료:", output_path)


## mock 대체 경로

앱이 열리지 않아도 `run_smoke_test()` 결과와 제공 카드 템플릿으로 실습을 완료합니다.

## 최종 확인

- 처리할 문서가 한 종류인가?
- 오류 영향과 사람 검토자가 적혀 있는가?
- 저장 위치와 삭제 시점이 적혀 있는가?
